# Golestan — Instructor Solution (Phase-by-Phase)

**Reference:** [Quera 108666](https://quera.org/problemset/108666)  
**Aligned with:** `golestan_oop_project_by_phases.html`

**Teaching plan:** Build the system in three phases. Run each phase's checks before continuing. Keep the notebook cells in order. All explanatory text and code comments are in English. Class attributes, `@classmethod`, and `@staticmethod` are not required.

> In this notebook, phase methods are attached to `Golestan` as we progress. This allows each phase to be taught and tested independently without repeating the entire class definition. For a final project submission, students should put these methods inside the `Golestan` class.


## Phase 1 — Register students, professors, and courses

**Required `Golestan` methods:**

- `register_student(name, identical_num, entering_year, field)`
- `register_professor(name, identical_num, field)`
- `make_class(name, class_id, field)`

Start by identifying the responsibilities of the three entity classes. Read-only `@property` accessors expose public values while internal lists remain mutable only through system operations.

In [ ]:
class Student:
    def __init__(self, name, identical_num, entering_year, field):
        self._name = name
        self._identical_num = identical_num
        self._entering_year = entering_year
        self._field = field
        self._courses = []

    @property
    def name(self):
        return self._name

    @property
    def identical_num(self):
        return self._identical_num

    @property
    def entering_year(self):
        return self._entering_year

    @property
    def field(self):
        return self._field

    @property
    def courses(self):
        # Expose a snapshot, not the mutable internal list.
        return tuple(self._courses)


class Professor:
    def __init__(self, name, identical_num, field):
        self._name = name
        self._identical_num = identical_num
        self._field = field
        self._courses = []

    @property
    def name(self):
        return self._name

    @property
    def identical_num(self):
        return self._identical_num

    @property
    def field(self):
        return self._field

    @property
    def courses(self):
        return tuple(self._courses)


class Course:
    def __init__(self, name, class_id, field):
        self._name = name
        self._class_id = class_id
        self._field = field
        self._professor = None
        self._students = []
        self._marks = {}  # national ID -> latest final mark

    @property
    def name(self):
        return self._name

    @property
    def class_id(self):
        return self._class_id

    @property
    def field(self):
        return self._field

    @property
    def professor(self):
        return self._professor

    @property
    def students(self):
        return tuple(self._students)

    def mark_of(self, student_id):
        return self._marks.get(student_id)


### Phase 1 — System class and registration methods

National IDs are unique across students and professors. Python dictionaries preserve insertion order. The exact success and failure strings form part of the public contract.

In [ ]:
class Golestan:
    def __init__(self):
        self._students = {}     # national ID -> Student (in insertion order)
        self._professors = {}   # national ID -> Professor
        self._courses = {}      # course ID -> Course

    @property
    def total_students(self):
        return len(self._students)

    @property
    def total_professors(self):
        return len(self._professors)

    @property
    def total_courses(self):
        return len(self._courses)

    def register_student(self, name, identical_num, entering_year, field):
        if identical_num in self._students or identical_num in self._professors:
            return 'this identical number previously registered'
        self._students[identical_num] = Student(name, identical_num, int(entering_year), field)
        return 'welcome to golestan'

    def register_professor(self, name, identical_num, field):
        if identical_num in self._students or identical_num in self._professors:
            return 'this identical number previously registered'
        self._professors[identical_num] = Professor(name, identical_num, field)
        return 'welcome to golestan'

    def make_class(self, name, class_id, field):
        if class_id in self._courses:
            return 'this class id previously used'
        self._courses[class_id] = Course(name, class_id, field)
        return 'class added successfully'


### Phase 1 — Verify registration behavior

In [ ]:
system = Golestan()
assert system.register_student('ali', '0012233445', 1398, 'software') == 'welcome to golestan'
assert system.register_professor('ahmadi', '0014433225', 'software') == 'welcome to golestan'
assert system.register_student('wrong', '0014433225', 1398, 'software') == 'this identical number previously registered'
assert system.make_class('riaziyek', '5552255522', 'software') == 'class added successfully'
assert system.total_students == 1
print('Phase 1 checks passed')

### Phase 1 — Mini unit tests

Ask learners to predict the results before running. These checks intentionally use separate `Golestan` objects to avoid shared test state.

In [ ]:
import unittest

class TestPhase1(unittest.TestCase):
    def setUp(self):
        self.g = Golestan()

    def test_new_student(self):
        result = self.g.register_student('ali', '0000000001', 1400, 'cs')
        self.assertEqual(result, 'welcome to golestan')
        self.assertEqual(self.g.total_students, 1)

    def test_duplicate_national_id_across_roles(self):
        self.g.register_student('ali', '0000000001', 1400, 'cs')
        self.assertEqual(self.g.register_professor('reza', '0000000001', 'cs'),
                         'this identical number previously registered')

    def test_read_only_property(self):
        self.g.register_student('ali', '0000000001', 1400, 'cs')
        with self.assertRaises(AttributeError):
            self.g._students['0000000001'].name = 'newname'

unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestPhase1)
)

## Phase 2 — Enrollment, professor assignment, and status reports

**Required `Golestan` methods:**

- `add_student(identical_num, class_id)`
- `add_professor(identical_num, class_id)`
- `student_status(identical_num)`
- `professor_status(identical_num)`
- `class_status(class_id)`

The order of error checks matters. Link existing student, professor, and course objects; do not create duplicate entities during enrollment.

In [ ]:
def add_student(self, identical_num, class_id):
    if identical_num not in self._students:
        return 'invalid student'
    if class_id not in self._courses:
        return 'invalid class'
    student = self._students[identical_num]
    course = self._courses[class_id]
    if student.field != course.field:
        return 'student field is not match'
    if student in course._students:
        return 'student is already registered'
    course._students.append(student)
    student._courses.append(course)
    return 'student added successfully to the class'


def add_professor(self, identical_num, class_id):
    if identical_num not in self._professors:
        return 'invalid professor'
    if class_id not in self._courses:
        return 'invalid class'
    professor = self._professors[identical_num]
    course = self._courses[class_id]
    if professor.field != course.field:
        return 'professor field is not match'
    if course.professor is not None:
        return 'this class has a professor'
    course._professor = professor
    professor._courses.append(course)
    return 'professor added successfully to the class'


def student_status(self, identical_num):
    student = self._students.get(identical_num)
    if student is None:
        return 'invalid student'
    if not student.courses:
        return ''  # The original question specifies no output in this case.
    return ' '.join([student.name, str(student.entering_year), student.field] +
                    [course.name for course in student.courses])


def professor_status(self, identical_num):
    professor = self._professors.get(identical_num)
    if professor is None:
        return 'invalid professor'
    if not professor.courses:
        return ''
    return ' '.join([professor.name, professor.field] +
                    [course.name for course in professor.courses])


def class_status(self, class_id):
    course = self._courses.get(class_id)
    if course is None:
        return 'invalid class'
    professor_name = course.professor.name if course.professor else 'None'
    names = [student.name for student in course.students]
    return ' '.join([professor_name] + names)


# Attach Phase 2 methods to the Phase 1 class to keep live-teaching cells short.
Golestan.add_student = add_student
Golestan.add_professor = add_professor
Golestan.student_status = student_status
Golestan.professor_status = professor_status
Golestan.class_status = class_status

### Phase 2 — Verify registration and report order

In [ ]:
assert system.add_student('0012233445', '5552255522') == 'student added successfully to the class'
assert system.add_student('0012233445', '5552255522') == 'student is already registered'
assert system.add_professor('0014433225', '5552255522') == 'professor added successfully to the class'
assert system.student_status('0012233445') == 'ali 1398 software riaziyek'
assert system.class_status('5552255522') == 'ahmadi ali'
print('Phase 2 checks passed')

### Phase 2 — Mini unit tests

In [ ]:
class TestPhase2(unittest.TestCase):
    def setUp(self):
        self.g = Golestan()
        self.g.register_student('ali', '0000000001', 1400, 'cs')
        self.g.register_professor('reza', '0000000002', 'cs')
        self.g.make_class('python', '1111111111', 'cs')

    def test_enrollment_and_duplicate(self):
        self.assertEqual(self.g.add_student('0000000001', '1111111111'),
                         'student added successfully to the class')
        self.assertEqual(self.g.add_student('0000000001', '1111111111'),
                         'student is already registered')

    def test_class_report(self):
        self.g.add_student('0000000001', '1111111111')
        self.g.add_professor('0000000002', '1111111111')
        self.assertEqual(self.g.class_status('1111111111'), 'reza ali')

unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestPhase2)
)

## Phase 3 — Marks, averages, and ranking

**Required `Golestan` methods:**

- `set_final_mark(professor_identical_num, student_identical_num, class_id, mark)`
- `mark_student(identical_num, class_id)`
- `mark_list(class_id)`
- `average_mark_professor(identical_num)`
- `average_mark_student(identical_num)`
- `top_student(field, entering_year)`
- `top_mark(class_id)`

A mark belongs to a **student–course relationship**, rather than to the student alone. Replacing a mark should not add a second mark for the same course. Average results are formatted with two decimals.

**Clarification in the educational specification:** Students without recorded marks are excluded from `top_student` ranking.

In [ ]:
def set_final_mark(self, professor_identical_num, student_identical_num, class_id, mark):
    if professor_identical_num not in self._professors:
        return 'invalid professor'
    if student_identical_num not in self._students:
        return 'invalid student'
    if class_id not in self._courses:
        return 'invalid class'
    professor = self._professors[professor_identical_num]
    student = self._students[student_identical_num]
    course = self._courses[class_id]
    if course.professor is not professor:
        return 'professor class is not match'
    if student not in course._students:
        return 'student did not registered'
    course._marks[student_identical_num] = int(mark)
    return 'student final mark added or changed'


def mark_student(self, identical_num, class_id):
    if identical_num not in self._students:
        return 'invalid student'
    if class_id not in self._courses:
        return 'invalid class'
    student = self._students[identical_num]
    course = self._courses[class_id]
    if student not in course._students:
        return 'student did not registered'
    value = course.mark_of(identical_num)
    return 'None' if value is None else str(value)


def mark_list(self, class_id):
    if class_id not in self._courses:
        return 'invalid class'
    course = self._courses[class_id]
    if course.professor is None:
        return 'no professor'
    if not course.students:
        return 'no student'
    return ' '.join('None' if course.mark_of(s.identical_num) is None
                    else str(course.mark_of(s.identical_num)) for s in course.students)


def average_mark_professor(self, identical_num):
    professor = self._professors.get(identical_num)
    if professor is None:
        return 'invalid professor'
    marks = [value for course in professor.courses for value in course._marks.values()]
    return f'{sum(marks) / len(marks):.2f}' if marks else 'None'


def average_mark_student(self, identical_num):
    student = self._students.get(identical_num)
    if student is None:
        return 'invalid student'
    marks = [course.mark_of(identical_num) for course in student.courses]
    marks = [mark for mark in marks if mark is not None]
    return f'{sum(marks) / len(marks):.2f}' if marks else 'None'


def top_student(self, field, entering_year):
    eligible = [s for s in self._students.values()
                if s.field == field and s.entering_year == int(entering_year)]
    if not eligible:
        return 'None'
    # The original question does not specify how to rank an entirely ungraded
    # eligible student against a graded student. This adaptation ranks only
    # students with at least one mark, returning None if nobody is graded.
    scored = []
    for student in eligible:
        marks = [c.mark_of(student.identical_num) for c in student.courses]
        marks = [m for m in marks if m is not None]
        if marks:
            scored.append((sum(marks) / len(marks), student))
    if not scored:
        return 'None'
    # max() keeps the first student on a tie (insertion order).
    return max(scored, key=lambda pair: pair[0])[1].name


def top_mark(self, class_id):
    if class_id not in self._courses:
        return 'invalid class'
    marks = self._courses[class_id]._marks.values()
    return str(max(marks)) if marks else 'None'


for name in ('set_final_mark', 'mark_student', 'mark_list',
             'average_mark_professor', 'average_mark_student',
             'top_student', 'top_mark'):
    setattr(Golestan, name, globals()[name])

### Phase 3 — Check grade updates and averages

In [ ]:
grades = Golestan()
grades.register_student('ali', '0000000001', 1400, 'cs')
grades.register_professor('reza', '0000000002', 'cs')
grades.make_class('python', '1111111111', 'cs')
assert grades.add_student('0000000001', '1111111111') == 'student added successfully to the class'
assert grades.add_professor('0000000002', '1111111111') == 'professor added successfully to the class'
assert grades.set_final_mark('0000000002', '0000000001', '1111111111', 16) == 'student final mark added or changed'
assert grades.set_final_mark('0000000002', '0000000001', '1111111111', 18) == 'student final mark added or changed'
assert grades.average_mark_student('0000000001') == '18.00'
assert grades.top_mark('1111111111') == '18'
print('Phase 3 checks passed')

### Phase 3 — Mini unit tests

In [ ]:
class TestPhase3(unittest.TestCase):
    def setUp(self):
        self.g = Golestan()
        self.g.register_student('ali', '0000000001', 1400, 'cs')
        self.g.register_professor('reza', '0000000002', 'cs')
        self.g.make_class('python', '1111111111', 'cs')
        self.g.add_student('0000000001', '1111111111')
        self.g.add_professor('0000000002', '1111111111')

    def test_mark_replacement(self):
        self.g.set_final_mark('0000000002', '0000000001', '1111111111', 13)
        self.g.set_final_mark('0000000002', '0000000001', '1111111111', 19)
        self.assertEqual(self.g.average_mark_student('0000000001'), '19.00')
        self.assertEqual(self.g.top_mark('1111111111'), '19')

    def test_invalid_professor_priority(self):
        self.assertEqual(
            self.g.set_final_mark('9999999999', '9999999998', '9999999997', 15),
            'invalid professor'
        )

unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestPhase3)
)

## Final integration — Text command adapter

Keep the command adapter separate from the domain model. The adapter parses lines, dispatches to public `Golestan` methods, and collects nonempty responses. It is added **only after all three phases work**.

In [ ]:
def execute_commands(lines):
    system = Golestan()
    result = []
    for line in lines:
        pieces = line.strip().split()
        if not pieces:
            continue
        command, *args = pieces
        if command == 'end':
            break
        if command == 'register_student':
            response = system.register_student(*args)
        elif command == 'register_professor':
            response = system.register_professor(*args)
        elif command == 'make_class':
            response = system.make_class(*args)
        elif command == 'add_student':
            response = system.add_student(*args)
        elif command == 'add_professor':
            response = system.add_professor(*args)
        elif command == 'student_status':
            response = system.student_status(*args)
        elif command == 'professor_status':
            response = system.professor_status(*args)
        elif command == 'class_status':
            response = system.class_status(*args)
        elif command == 'set_final_mark':
            response = system.set_final_mark(*args)
        elif command == 'mark_student':
            response = system.mark_student(*args)
        elif command == 'mark_list':
            response = system.mark_list(*args)
        elif command == 'average_mark_professor':
            response = system.average_mark_professor(*args)
        elif command == 'average_mark_student':
            response = system.average_mark_student(*args)
        elif command == 'top_student':
            response = system.top_student(*args)
        elif command == 'top_mark':
            response = system.top_mark(*args)
        else:
            raise ValueError(f'Unknown command: {command}')
        if response != '':
            result.append(response)
    return result

## Integration check — Official sample inputs

In [ ]:
SAMPLE_1_INPUT = 'register_student ali 0012233445 1398 software\nregister_student alireza 0012233445 1397 cs\nregister_student erfan 0043332227 1400 software\nregister_professor ahmadi 0014433225 software\nmake_class riaziyek 9876543210 math\nmake_class riaziyek 5552255522 software\nadd_student 0012233444 5552255522\nadd_student 0012233445 5552255522\nadd_professor 0014433225 5552255522\nstudent_status 0012233445\nadd_student 0043332227 9876543210\nadd_student 0043332227 5552255522\nstudent_status 0043332227\nprofessor_status 0014433225\nclass_status 5552255522\nset_final_mark 0014433225 0012233445 5552255522 18\nmark_student 0012233445 5552255522\nmark_student 0043332227 5552255522\nset_final_mark 0014433225 0043332227 5552255522 19\nmark_student 0012233445 5552255522\nmark_student 0043332227 5552255522\nmark_list 5552255522\ntop_mark 5552255522\ntop_student software 1398\ntop_student software 1401\nend'
SAMPLE_1_EXPECTED = 'welcome to golestan\nthis identical number previously registered\nwelcome to golestan\nwelcome to golestan\nclass added successfully\nclass added successfully\ninvalid student\nstudent added successfully to the class\nprofessor added successfully to the class\nali 1398 software riaziyek\nstudent field is not match\nstudent added successfully to the class\nerfan 1400 software riaziyek\nahmadi software riaziyek\nahmadi ali erfan\nstudent final mark added or changed\n18\nNone\nstudent final mark added or changed\n18\n19\n18 19\n19\nali\nNone'
SAMPLE_2_INPUT = 'register_student karimi 0023333334 1405 math\nregister_professor hajjarian 9987777776 math\nregister_professor azari 8876666665 math\nmake_class adadi 6222816098 math\nmake_class jabr 6222816099 math\nadd_professor 9987777776 6222816098\nadd_professor 8876666665 6222816099\nadd_professor 9987777776 6222816099\nadd_student 0023333334 6222816098\nadd_student 0023333334 6222816099\nset_final_mark 9987777776 0023333334 6222816098 19\nset_final_mark 9987777776 0023333334 6222816099 18\nset_final_mark 8876666665 0023333334 6222816099 17\nset_final_mark 8876666665 0023333334 6222816099 13\nset_final_mark 9987777776 0023333334 6222816098 11\naverage_mark_professor 9987777776\naverage_mark_professor 8876666665\naverage_mark_student 0023333334\nmark_list 6222816098\nmark_list 6222816099\nstudent_status 0023333334\nprofessor_status 9987777776\nprofessor_status 8876666665\nclass_status 6222816098\nclass_status 6222816099\nend'
SAMPLE_2_EXPECTED = 'welcome to golestan\nwelcome to golestan\nwelcome to golestan\nclass added successfully\nclass added successfully\nprofessor added successfully to the class\nprofessor added successfully to the class\nthis class has a professor\nstudent added successfully to the class\nstudent added successfully to the class\nstudent final mark added or changed\nprofessor class is not match\nstudent final mark added or changed\nstudent final mark added or changed\nstudent final mark added or changed\n11.00\n13.00\n12.00\n11\n13\nkarimi 1405 math adadi jabr\nhajjarian math adadi\nazari math jabr\nhajjarian karimi\nazari karimi'

for index, (source, expected) in enumerate([
    (SAMPLE_1_INPUT, SAMPLE_1_EXPECTED),
    (SAMPLE_2_INPUT, SAMPLE_2_EXPECTED)
], start=1):
    actual = execute_commands(source.splitlines())
    expected_lines = expected.splitlines()
    assert actual == expected_lines, f'Sample {index} differs: {actual}'
    print(f'Official sample {index}: PASS ({len(actual)} output lines)')

## Final teacher checklist

1. Run the notebook from the first cell through the final sample checks.
2. Ask which class owns each piece of data and why.
3. Demonstrate that read-only properties reject direct assignment.
4. Have students write a new `unittest` test before changing the implementation.
5. Emphasize that matching sample output does not replace unit testing.
6. For student submissions, move phase methods into the `Golestan` class instead of attaching them dynamically.
